In [31]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
import numpy as np

In [33]:
import pandas as pd
import utils.experiments as exp_utils
import lingam



In [57]:
def dag_sparsity(adj):
    p = adj.shape[0]
    # possible edges in a DAG
    possible_edges = p * (p - 1) / 2
    edges = np.count_nonzero(adj)
    
    density = edges / possible_edges
    sparsity = 1 - density

    
    return {
        "edges": edges,
        "density": density,
        "sparsity": sparsity
    }


"""
Convert a saved LiNGAM adjacency .npz into Gephi-ready nodes.csv and edges.csv.

Expected NPZ keys:
- adjacency: (p, p) numpy array, where adjacency[i, j] = effect of j -> i
- features:  (p,) array of feature names in the same order as adjacency indices
"""

from __future__ import annotations

import os
def npz_to_gephi(
    npz_path: str,
    outdir: str = "",
    nodes_filename: str = "nodes.csv",
    edges_filename: str = "edges.csv",
    weight_threshold: float | None = None,
    absolute_threshold: bool = True,
    keep_self_loops: bool = False,
) -> tuple[str, str]:
    """
    Load adjacency + feature names from NPZ and export Gephi nodes + edges CSVs.

    Parameters
    ----------
    npz_path : str
        Path to the saved .npz file (created via np.savez_compressed).
    outdir : str
        Output directory for Gephi CSV files.
    nodes_filename : str
        Name of nodes CSV file.
    edges_filename : str
        Name of edges CSV file.
    weight_threshold : float | None
        If set, keep only edges with |weight| >= threshold (default uses abs if absolute_threshold=True).
        Useful to reduce graph size for visualization.
    absolute_threshold : bool
        If True, threshold is applied on abs(weight); else on weight directly.
    keep_self_loops : bool
        If True, keep i->i edges (usually False).

    Returns
    -------
    (nodes_path, edges_path) : tuple[str, str]
        Paths to the exported nodes and edges CSV files.
    """
    if not os.path.isfile(npz_path):
        raise FileNotFoundError(f"NPZ file not found: {npz_path}")

    os.makedirs(outdir, exist_ok=True)

    data = np.load(npz_path, allow_pickle=True)
    if "adjacency" not in data or "features" not in data:
        raise ValueError("NPZ must contain keys: 'adjacency' and 'features'.")

    A = data["adjacency"]
    features = data["features"].tolist()

    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError(f"Adjacency must be square; got shape {A.shape}.")
    if len(features) != A.shape[0]:
        raise ValueError(
            f"features length ({len(features)}) must match adjacency size ({A.shape[0]})."
        )

    p = A.shape[0]

    # -----------------------------
    # Nodes file (Gephi)
    # -----------------------------
    # Gephi nodes CSV minimal columns: Id, Label
    # We'll use feature name as both Id and Label to keep things simple.
    nodes_df = pd.DataFrame({"Id": features, "Label": features})
    nodes_path = os.path.join(outdir, nodes_filename)
    nodes_df.to_csv(nodes_path, index=False)

    # -----------------------------
    # Edges file (Gephi)
    # -----------------------------
    # LiNGAM convention: A[i, j] = effect of j -> i
    # So edge Source = features[j], Target = features[i]
    rows, cols = np.where(A != 0)

    if not keep_self_loops:
        mask = rows != cols
        rows, cols = rows[mask], cols[mask]

    weights = A[rows, cols]

    if weight_threshold is not None:
        if absolute_threshold:
            keep = np.abs(weights) >= weight_threshold
        else:
            keep = weights >= weight_threshold
        rows, cols, weights = rows[keep], cols[keep], weights[keep]

    edges_df = pd.DataFrame(
        {
            "Source": [features[j] for j in cols],
            "Target": [features[i] for i in rows],
            "Weight": weights,
        }
    )

    # Optional: add AbsWeight for sorting/debugging
    edges_df["AbsWeight"] = np.abs(edges_df["Weight"])
    edges_df = edges_df.sort_values("AbsWeight", ascending=False)

    edges_path = os.path.join(outdir, edges_filename)
    edges_df.to_csv(edges_path, index=False)

    return nodes_path, edges_path





In [39]:
df = pd.read_csv("real_data_application/exp4/data/clr_transformation_cmd_healthy_no_standardization_30percent_presence.csv")

In [40]:
df.head()

,clr_Adlercreutzia_equolifaciens,clr_Agathobaculum_butyriciproducens,clr_Akkermansia_muciniphila,clr_Alistipes_finegoldii,clr_Alistipes_indistinctus,clr_Alistipes_putredinis,clr_Alistipes_shahii,clr_Anaeromassilibacillus_sp_An250,clr_Anaerostipes_hadrus,clr_Anaerotruncus_colihominis,...,clr_Ruminococcus_lactaris,clr_Ruminococcus_torques,clr_Ruthenibacterium_lactatiformans,clr_Slackia_isoflavoniconvertens,clr_Streptococcus_parasanguinis,clr_Streptococcus_salivarius,clr_Streptococcus_thermophilus,clr_Turicibacter_sanguinis,clr_Turicimonas_muris,clr_Veillonella_dispar
0,-5.464890,3.066374,-5.464890,2.505772,-5.464890,5.147224,-5.464890,-5.464890,1.667912,-3.061302,...,3.399377,3.559675,1.905144,1.644100,-0.726882,2.703447,0.053188,0.236523,-5.464890,-5.464890
1,-1.290349,2.516752,5.128229,2.421766,-0.329438,4.689847,2.772765,1.420548,1.345095,-6.053658,...,-6.053658,3.374222,0.636684,-6.053658,-0.560118,1.589196,0.790473,-3.417372,-6.053658,-6.053658
2,-0.165846,3.459044,2.675797,3.740706,1.075613,4.422126,0.008892,-2.374964,2.370562,-6.415169,...,1.947436,2.876105,0.463376,-0.654284,-0.540451,-0.029667,-2.330786,1.169906,-1.248328,-1.315173
3,1.447211,4.588163,-5.882334,1.635711,-5.882334,4.590539,3.579751,-5.882334,4.197855,-5.882334,...,2.344517,2.635034,-0.219288,2.325768,-5.882334,-2.414129,-5.882334,-5.882334,-1.955084,-1.502623
4,0.631569,2.333622,4.831999,0.314348,-0.723819,5.597243,2.553661,-2.448725,3.683351,1.529562,...,2.621451,2.989213,1.971545,-6.197172,0.164792,0.864316,0.733321,-6.197172,-6.197172,-6.197172


In [41]:
n_samples = len(df.columns)*3

In [42]:
df_sub = df.sample(n=n_samples, replace=False, random_state=42)

In [60]:
feature_names = df_sub.columns.to_list()

In [43]:
X = df_sub.values

In [44]:
lingam_model = lingam.DirectLiNGAM()
lingam_model.fit(X)

In [45]:
adj_matrix = lingam_model.adjacency_matrix_

In [46]:
causal_order = lingam_model.causal_order_

In [47]:
ebic_estimation = exp_utils.estimate_adjacency_matrix_ebic(X, causal_order)

In [52]:
np.count_nonzero(ebic_estimation)

378

In [53]:
np.count_nonzero(adj_matrix)

746

In [55]:
dag_sparsity(ebic_estimation)

{'edges': 378, 'density': 0.07636363636363637, 'sparsity': 0.9236363636363636}

In [56]:
dag_sparsity(adj_matrix)

{'edges': 746, 'density': 0.1507070707070707, 'sparsity': 0.8492929292929293}

In [68]:
adj_matrix.shape

(100, 100)

In [61]:
# 1) Save adjacency + feature names efficiently (compressed binary)
np.savez_compressed(
    "./real_data_application/exp4/output/lingam/lingam_adj_matrix.npz",
    adjacency=adj_matrix.astype(np.float32),
    features=np.array(feature_names, dtype=object),
    )
np.savez_compressed(
    "./real_data_application/exp4/output/eBIC/lingam_adj_matrix.npz",
    adjacency=ebic_estimation.astype(np.float32),
    features=np.array(feature_names, dtype=object),
    )

In [65]:
nodes_path, edges_path = npz_to_gephi(
                                      npz_path="./real_data_application/exp4/output/eBIC/lingam_adj_matrix.npz",
                                      outdir="./real_data_application/exp4/output/eBIC/",
                                      # Set a threshold if you want fewer edges in Gephi, e.g. 0.05 or 0.1
                                      weight_threshold=None,
                                      absolute_threshold=True,
                                      keep_self_loops=False,
                                      )